In [ ]:
# Colab (Python 3.12) setup:
# 1) Mount Google Drive
# 2) Clone RDT repo
# 3) Install deps (minimal)
# 4) Download/load T5-XXL via the repo's T5Embedder
# 5) Compute + save embeddings for your 5 instructions
# 6) Store embeddings folder on Google Drive and show how to load it

import os
import shutil
from pathlib import Path

In [ ]:
# ----------------------------
# 0) Google Drive mount
# ----------------------------
from google.colab import drive
drive.mount("/content/drive")

# Change this if you want a different location on Drive
DRIVE_OUT_DIR = Path("/content/drive/MyDrive/rdt_lang_embeddings_maniskill")
DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional: if you use gated HF models, set token in Colab Secrets or here:
# os.environ["HUGGINGFACE_HUB_TOKEN"] = "hf_..."

In [ ]:
# ----------------------------
# 1) Clone repo
# ----------------------------
REPO_DIR = Path("/content/RoboticsDiffusionTransformer")
if not REPO_DIR.exists():
    !git clone https://github.com/thu-ml/RoboticsDiffusionTransformer.git /content/RoboticsDiffusionTransformer

os.chdir(str(REPO_DIR))

In [ ]:
# ----------------------------
# 2) Install dependencies
# ----------------------------
# Notes:
# - Colab already has torch; we try to keep changes minimal.
# - If you need a specific torch version, pin it here (may take time and can break other libs).
!pip -q install --upgrade pip
!pip -q install packaging==24.0 pyyaml huggingface_hub safetensors sentencepiece
!pip -q install -r requirements.txt

# flash-attn often fails on Colab; only try if you know it works in your runtime:
# !pip -q install flash-attn --no-build-isolation

In [ ]:
# ----------------------------
# 3) Compute embeddings
# ----------------------------
import torch
import yaml

from models.multimodal_encoder.t5_encoder import T5Embedder

GPU = 0
MODEL_PATH = "google/t5-v1_1-xxl"
CONFIG_PATH = "configs/base.yaml"

# Local scratch output; we'll copy to Drive at the end
LOCAL_SAVE_DIR = Path("/content/rdt_lang_embeddings_tmp")
if LOCAL_SAVE_DIR.exists():
    shutil.rmtree(LOCAL_SAVE_DIR)
LOCAL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Offload directory (recommended if VRAM < 24GB)
OFFLOAD_DIR = Path("/content/rdt_t5_offload")
OFFLOAD_DIR.mkdir(parents=True, exist_ok=True)

TASKS = [
    ("PushCube-v1", "Push the blue cube towards the target"),
    ("PickCube-v1", "Pick up the red cube and bring it to the green goal"),
    ("StackCube-v1", "Pick up the red cube and stack it on top of the green cube"),
    ("PegInsertionSide-v1", "Pick up the peg and insert it into the hole"),
    ("PullCubeTool-v1", "Use the hook tool to pull the blue cube towards the robot arm base"),
]

with open(CONFIG_PATH, "r") as fp:
    config = yaml.safe_load(fp)

device = torch.device(f"cuda:{GPU}" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Create embedder (this will download weights from HF if not present)
text_embedder = T5Embedder(
    from_pretrained=MODEL_PATH,
    model_max_length=config["dataset"]["tokenizer_max_length"],
    device=device,
    use_offload_folder=str(OFFLOAD_DIR) if OFFLOAD_DIR is not None else None,
)
tokenizer, text_encoder = text_embedder.tokenizer, text_embedder.model
text_encoder.eval()

def encode_one(name: str, instruction: str) -> Path:
    tokens = tokenizer(
        instruction,
        return_tensors="pt",
        padding="longest",
        truncation=True,
    )["input_ids"].to(device)

    tokens = tokens.view(1, -1)
    with torch.no_grad():
        emb = text_encoder(tokens).last_hidden_state.detach().cpu()

    save_path = LOCAL_SAVE_DIR / f"{name}.pt"
    torch.save(
        {
            "name": name,
            "instruction": instruction,
            "embeddings": emb,
        },
        save_path,
    )
    print(f"[OK] {name}: {emb.shape} -> {save_path}")
    return save_path

saved = []
for name, instr in TASKS:
    saved.append(encode_one(name, instr))

# Optional: save an index file for convenience
index_path = LOCAL_SAVE_DIR / "index.txt"
with open(index_path, "w", encoding="utf-8") as f:
    for name, instr in TASKS:
        f.write(f"{name}\t{instr}\n")
print(f"[OK] Wrote index: {index_path}")

In [ ]:
!ls -lh /content/rdt_lang_embeddings_tmp


In [ ]:
!cat /content/rdt_lang_embeddings_tmp/index.txt


In [ ]:
# ----------------------------
# 4) Copy embeddings folder to Google Drive
# ----------------------------
# Copy the whole folder to Drive (overwrite existing)
if DRIVE_OUT_DIR.exists():
    shutil.rmtree(DRIVE_OUT_DIR)
shutil.copytree(LOCAL_SAVE_DIR, DRIVE_OUT_DIR)
print(f"[OK] Copied embeddings to Drive: {DRIVE_OUT_DIR}")

In [ ]:
# ----------------------------
# 5) Example: load embeddings folder from Drive
# ----------------------------
def load_all_embeddings(folder: Path):
    out = {}
    for pt in folder.glob("*.pt"):
        d = torch.load(pt, map_location="cpu")
        out[d["name"]] = d
    return out

loaded = load_all_embeddings(DRIVE_OUT_DIR)
print("[OK] Loaded keys:", list(loaded.keys()))
print("[Example] One entry shapes:", {k: v["embeddings"].shape for k, v in list(loaded.items())[:1]})